In [1]:
import pandas as pd
import os

# ====== 1. Danh sách file CIC-IDS2018 cần gộp ======
file_paths = [
    "../dataset/CSE-CIC-IDS2018/Bot.csv",
    "../dataset/CSE-CIC-IDS2018/DoS-attacks-Hulk.csv",
    "../dataset/CSE-CIC-IDS2018/DoS-attacks-SlowHTTPTest.csv"
]

# ====== 2. Hàm load từng file ======
def load_cic_file(path):
    ext = os.path.splitext(path)[1].lower()
    if ext == ".csv":
        df = pd.read_csv(path, low_memory=False)
    elif ext in [".xls", ".xlsx"]:
        df = pd.read_excel(path)
    else:
        raise ValueError(f"Không hỗ trợ định dạng: {ext}")
    
    df.columns = df.columns.str.strip()
    df = df.loc[:, ~df.columns.str.contains("^Unnamed")]
    df["__source__"] = os.path.basename(path)
    return df

# ====== 3. Gộp toàn bộ file ======
df_list = [load_cic_file(p) for p in file_paths]
df_all = pd.concat(df_list, ignore_index=True)

# ====== 4. Tổng hợp nhãn gốc trong cột Label ======
summary_total = df_all["Label"].value_counts().to_frame(name="Total")
summary_total["Percentage (%)"] = (summary_total["Total"] / len(df_all) * 100).round(2)

print("📊 Tổng hợp nhãn CIC-IDS2018 sau khi gộp nhiều file:\n")
print(summary_total)

# ====== 5. Tách dữ liệu để tạo file train/test theo Table 6 ======
# Dùng trực tiếp nhãn gốc y như file Label gốc
target_counts = {
    "Benign": {"train": 646603, "test": 161601},
    "Bot": {"train": 86811, "test": 21709},
    "DoS attacks-SlowHTTPTest": {"train": 34, "test": 7},
    "DoS attacks-Hulk": {"train": 87117, "test": 21811},
}

train_parts, test_parts = [], []

for label, counts in target_counts.items():
    df_label = df_all[df_all["Label"] == label]
    needed = counts["train"] + counts["test"]
    available = len(df_label)

    if available < needed:
        print(f"⚠️ Không đủ mẫu cho lớp '{label}': cần {needed}, chỉ có {available}. Bỏ qua.")
        continue

    df_sample = df_label.sample(n=needed, random_state=42)
    df_train = df_sample.iloc[:counts["train"]]
    df_test = df_sample.iloc[counts["train"]:]
    train_parts.append(df_train)
    test_parts.append(df_test)

# ====== 6. Gộp lại và lưu file ======
df_train_final = pd.concat(train_parts, ignore_index=True)
df_test_final = pd.concat(test_parts, ignore_index=True)

os.makedirs("../dataset/CSE-CIC-IDS2018", exist_ok=True)
df_train_final.to_csv("../dataset/CSE-CIC-IDS2018/cic2018_training-set.csv", index=False)
df_test_final.to_csv("../dataset/CSE-CIC-IDS2018/cic2018_testing-set.csv", index=False)

print("✅ Đã lưu:")
print("  - cic2018_training-set.csv")
print("  - cic2018_testing-set.csv")


📊 Tổng hợp nhãn CIC-IDS2018 sau khi gộp nhiều file:

                            Total  Percentage (%)
Label                                            
Benign                    2159395           70.86
DoS attacks-Hulk           461912           15.16
Bot                        286191            9.39
DoS attacks-SlowHTTPTest   139890            4.59
✅ Đã lưu:
  - cic2018_training-set.csv
  - cic2018_testing-set.csv


In [3]:
import pandas as pd

train_df = pd.read_csv("../dataset/CSE-CIC-IDS2018/cic2018_training-set.csv")
test_df = pd.read_csv("../dataset/CSE-CIC-IDS2018/cic2018_testing-set.csv")

print("📊 Nhãn train:")
print(train_df["Label"].value_counts())

print("\n📊 Nhãn test:")
print(test_df["Label"].value_counts())


📊 Nhãn train:
Label
Benign                      646603
DoS attacks-Hulk             87117
Bot                          86811
DoS attacks-SlowHTTPTest        34
Name: count, dtype: int64

📊 Nhãn test:
Label
Benign                      161601
DoS attacks-Hulk             21811
Bot                          21709
DoS attacks-SlowHTTPTest         7
Name: count, dtype: int64


In [5]:
if "__source__" in train_df.columns:
    print("\n📂 Nguồn train:")
    print(train_df["__source__"].value_counts())

    print("\n📂 Nguồn test:")
    print(test_df["__source__"].value_counts())




📂 Nguồn train:
__source__
DoS-attacks-Hulk.csv            416321
Bot.csv                         305873
DoS-attacks-SlowHTTPTest.csv     98371
Name: count, dtype: int64

📂 Nguồn test:
__source__
DoS-attacks-Hulk.csv            103776
Bot.csv                          76704
DoS-attacks-SlowHTTPTest.csv     24648
Name: count, dtype: int64
